# A2 — DCGAN baseline versus differentiable augmentation
## 1. Research protocol
Compare exactly `dcgan_baseline` and `dcgan_diffaugment`: identical initialization, sample order,
latent sequence, horizontal flips, optimizer, batch64 and 20,000 updates. Augmentation has isolated RNG.
Use final EMA0.999, never test-based selection. Validation supplies diagnostics only.
Primary paper: [Zhao et al., NeurIPS 2020](https://arxiv.org/abs/2006.10738).
Attach `splcher/animefacedataset` and select T4×2; each variant has one GPU.
This full-data, single-seed study cannot establish significance or data efficiency at other data scales.


### Environment

**Purpose:** Keep existing torch/CUDA; install only missing auxiliaries. CPU smoke is supported.

**Inputs:** Python environment and writable working directory; Internet only for missing packages.

**Outputs:** RUNTIME directory and dependency readiness.


In [ ]:
import sys, subprocess, importlib.util, json, os, time, shutil, zipfile
from pathlib import Path
for module,package in [('PIL','pillow'),('numpy','numpy'),('scipy','scipy'),('matplotlib','matplotlib')]:
    if importlib.util.find_spec(module) is None: subprocess.check_call([sys.executable,'-m','pip','install',package])
RUNTIME=Path('/kaggle/working/a2_diffaugment_runtime' if Path('/kaggle').exists() else './a2_diffaugment_runtime')
RUNTIME.mkdir(parents=True,exist_ok=True)
import torch
if not hasattr(torch.amp,'GradScaler'):
    raise RuntimeError('PyTorch >=2.3 is required. Select a recent Kaggle image; do not replace CUDA packages in a running kernel.')

### Shared data utilities

**Purpose:** Decode RGB, remove corrupt files, deduplicate before split and cache uint8 64×64 images. Exact image duplicates cannot cross splits. Test is held out; train uses deterministic horizontal flips.

**Inputs:** Embedded implementation below; RUNTIME is the destination directory.

**Outputs:** An overwritten, numbered worker fragment; execute the assembly cell after all fragments.


In [ ]:
%%writefile {RUNTIME}/00.py

import os, sys, json, time, random, hashlib, signal, threading, copy, math, csv
from pathlib import Path
import numpy as np
from PIL import Image, ImageOps
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader



def digest(value):
    return hashlib.sha256(json.dumps(value,sort_keys=True).encode()).hexdigest()

def atomic_json(value,path):
    path=Path(path); path.parent.mkdir(parents=True,exist_ok=True)
    temp=path.with_suffix('.tmp'); temp.write_text(json.dumps(value,indent=2)); temp.replace(path)

def seed_all(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark=False
    torch.backends.cudnn.deterministic=True

def prepare(c):
    root=Path(c['data']['root']) if c['data']['root'] else None
    if root is None:
        candidates=list(Path('/kaggle/input').glob('**/images'))
        candidates=[p for p in candidates if any(p.glob('*.jpg'))]
        if len(candidates)!=1: raise ValueError('Set data.root explicitly to the anime images directory: '+str(candidates))
        root=candidates[0]
    if (root/'images').is_dir(): root=root/'images'
    files=sorted(p for p in root.rglob('*') if p.suffix.lower() in ('.jpg','.jpeg','.png'))
    if not files: raise FileNotFoundError('No images under '+str(root))
    cache=Path(c['data']['cache_root']); cache.mkdir(parents=True,exist_ok=True)
    records=[]; rejected=[]; duplicates=[]; seen=set()
    for p in files:
        try:
            with Image.open(p) as im:
                im=im.convert('RGB'); size=im.size
                sha=hashlib.sha256(str(size).encode()+im.tobytes()).hexdigest()
            if sha in seen: duplicates.append(str(p.relative_to(root))); continue
            seen.add(sha); records.append({'filename':str(p.relative_to(root)),'sha256':sha,'width':size[0],'height':size[1]})
        except (OSError,ValueError) as error: rejected.append({'filename':str(p),'error':str(error)})
    n=len(records); v=c['data']['val_size']; t=c['data']['test_size']
    if n<=v+t+64: raise ValueError('Not enough unique valid images for configured splits')
    order=np.random.default_rng(c['seed']).permutation(n)
    for i in range(n): records[i]['split']='train'
    for i in order[:v]: records[int(i)]['split']='val'
    for i in order[v:v+t]: records[int(i)]['split']='test'
    h=digest({'records':records,'preprocessing':'rgb_fit_bilinear64_v1'}); array=cache/(h+'.npy')
    if not array.exists():
        temp=cache/(h+'.partial.npy')
        mmap=np.lib.format.open_memmap(temp,mode='w+',dtype=np.uint8,shape=(n,64,64,3))
        for i,row in enumerate(records):
            with Image.open(root/row['filename']) as im:
                mmap[i]=np.asarray(ImageOps.fit(im.convert('RGB'),(64,64),method=Image.Resampling.BILINEAR))
        mmap.flush(); del mmap; temp.replace(array)
    manifest={'hash':h,'array':str(array),'records':records,'corrupt':rejected,'duplicates':duplicates,
              'source':'splcher/animefacedataset','license_note':'Consult mounted Kaggle metadata and upstream redistribution concerns; no source images exported','attribution':'https://github.com/bchao1/Anime-Face-Dataset'}
    atomic_json(manifest,cache/'manifest.json')
    print('Prepared:',n,'unique images;',len(rejected),'corrupt;',len(duplicates),'duplicates; hash',h)
    return manifest

class Images(Dataset):
    def __init__(self,m,split,seed,epoch=0):
        self.array=np.load(m['array'],mmap_mode='r'); self.ids=[i for i,r in enumerate(m['records']) if r['split']==split]
        self.seed,self.epoch,self.train=seed,epoch,split=='train'
    def __len__(self): return len(self.ids)
    def __getitem__(self,i):
        index=self.ids[i]; x=np.array(self.array[index],copy=True)
        if self.train and int(digest([self.seed,self.epoch,index])[:8],16)%2: x=x[:,::-1].copy()
        return torch.from_numpy(x).permute(2,0,1).float()/127.5-1

def rng_state():
    return dict(python=random.getstate(),numpy=np.random.get_state(),torch=torch.get_rng_state(),cuda=torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None)

def restore_rng(s):
    random.setstate(s['python']); np.random.set_state(s['numpy']); torch.set_rng_state(s['torch'])
    if s['cuda'] is not None: torch.cuda.set_rng_state_all(s['cuda'])

class EMA:
    def __init__(self,model,decay): self.model=copy.deepcopy(model).eval().requires_grad_(False); self.decay=decay
    @torch.no_grad()
    def update(self,model):
        for a,b in zip(self.model.parameters(),model.parameters()): a.lerp_(b,1-self.decay)
        for a,b in zip(self.model.buffers(),model.buffers()): a.copy_(b)

### DCGAN architecture

**Purpose:** G maps (B,128,1,1) to (B,3,64,64); D emits logits. Non-saturating BCE uses logits without sigmoid. D parameters and BatchNorm stats are frozen in the G phase.

**Inputs:** Embedded implementation below; RUNTIME is the destination directory.

**Outputs:** An overwritten, numbered worker fragment; execute the assembly cell after all fragments.


In [ ]:
%%writefile {RUNTIME}/01.py
class Generator(nn.Module):
    def __init__(self,z=128,w=64):
        super().__init__(); layers=[]
        for a,b,k,s,p in [(z,8*w,4,1,0),(8*w,4*w,4,2,1),(4*w,2*w,4,2,1),(2*w,w,4,2,1)]:
            layers += [nn.ConvTranspose2d(a,b,k,s,p,bias=False),nn.BatchNorm2d(b),nn.ReLU(True)]
        self.net=nn.Sequential(*layers,nn.ConvTranspose2d(w,3,4,2,1,bias=False),nn.Tanh())
    def forward(self,z): return self.net(z)

class Discriminator(nn.Module):
    def __init__(self,w=64):
        super().__init__(); layers=[nn.Conv2d(3,w,4,2,1,bias=False),nn.LeakyReLU(.2,True)]
        for a,b in [(w,2*w),(2*w,4*w),(4*w,8*w)]:
            layers += [nn.Conv2d(a,b,4,2,1,bias=False),nn.BatchNorm2d(b),nn.LeakyReLU(.2,True)]
        self.net=nn.Sequential(*layers,nn.Conv2d(8*w,1,4,1,0,bias=False))
    def forward(self,x): return self.net(x).flatten()

def initialize(module):
    if isinstance(module,(nn.Conv2d,nn.ConvTranspose2d)): nn.init.normal_(module.weight,0,.02)
    elif isinstance(module,nn.BatchNorm2d): nn.init.normal_(module.weight,1,.02); nn.init.zeros_(module.bias)


# Source: https://github.com/mit-han-lab/data-efficient-gans/blob/master/DiffAugment_pytorch.py

### Licensed DiffAugment reference

**Purpose:** Author source, BSD-2-Clause notice and upstream SHA256 are embedded below. Color, translation and cutout apply to real/fake in D and fake in G; gradients pass through augmentation.

**Inputs:** Embedded implementation below; RUNTIME is the destination directory.

**Outputs:** An overwritten, numbered worker fragment; execute the assembly cell after all fragments.


In [ ]:
%%writefile {RUNTIME}/02.py

# Author DiffAugment snapshot SHA256 d61fed24900fea3be17e3cd10d8880f895905732fd53f1f9b3013673b36fa0e2
# Copyright (c) 2020, Shengyu Zhao, Zhijian Liu, Ji Lin, Jun-Yan Zhu, and Song Han
# All rights reserved.
# 
# Redistribution and use in source and binary forms, with or without
# modification, are permitted provided that the following conditions are met:
# 
# * Redistributions of source code must retain the above copyright notice, this
#   list of conditions and the following disclaimer.
# 
# * Redistributions in binary form must reproduce the above copyright notice,
#   this list of conditions and the following disclaimer in the documentation
#   and/or other materials provided with the distribution.
# 
# THIS SOFTWARE IS PROVIDED BY THE COPYRIGHT HOLDERS AND CONTRIBUTORS "AS IS"
# AND ANY EXPRESS OR IMPLIED WARRANTIES, INCLUDING, BUT NOT LIMITED TO, THE
# IMPLIED WARRANTIES OF MERCHANTABILITY AND FITNESS FOR A PARTICULAR PURPOSE ARE
# DISCLAIMED. IN NO EVENT SHALL THE COPYRIGHT HOLDER OR CONTRIBUTORS BE LIABLE
# FOR ANY DIRECT, INDIRECT, INCIDENTAL, SPECIAL, EXEMPLARY, OR CONSEQUENTIAL
# DAMAGES (INCLUDING, BUT NOT LIMITED TO, PROCUREMENT OF SUBSTITUTE GOODS OR
# SERVICES; LOSS OF USE, DATA, OR PROFITS; OR BUSINESS INTERRUPTION) HOWEVER
# CAUSED AND ON ANY THEORY OF LIABILITY, WHETHER IN CONTRACT, STRICT LIABILITY,
# OR TORT (INCLUDING NEGLIGENCE OR OTHERWISE) ARISING IN ANY WAY OUT OF THE USE
# OF THIS SOFTWARE, EVEN IF ADVISED OF THE POSSIBILITY OF SUCH DAMAGE.
# Differentiable Augmentation for Data-Efficient GAN Training
# Shengyu Zhao, Zhijian Liu, Ji Lin, Jun-Yan Zhu, and Song Han
# https://arxiv.org/pdf/2006.10738

import torch
import torch.nn.functional as F


def DiffAugment(x, policy='', channels_first=True):
    if policy:
        if not channels_first:
            x = x.permute(0, 3, 1, 2)
        for p in policy.split(','):
            for f in AUGMENT_FNS[p]:
                x = f(x)
        if not channels_first:
            x = x.permute(0, 2, 3, 1)
        x = x.contiguous()
    return x


def rand_brightness(x):
    x = x + (torch.rand(x.size(0), 1, 1, 1, dtype=x.dtype, device=x.device) - 0.5)
    return x


def rand_saturation(x):
    x_mean = x.mean(dim=1, keepdim=True)
    x = (x - x_mean) * (torch.rand(x.size(0), 1, 1, 1, dtype=x.dtype, device=x.device) * 2) + x_mean
    return x


def rand_contrast(x):
    x_mean = x.mean(dim=[1, 2, 3], keepdim=True)
    x = (x - x_mean) * (torch.rand(x.size(0), 1, 1, 1, dtype=x.dtype, device=x.device) + 0.5) + x_mean
    return x


def rand_translation(x, ratio=0.125):
    shift_x, shift_y = int(x.size(2) * ratio + 0.5), int(x.size(3) * ratio + 0.5)
    translation_x = torch.randint(-shift_x, shift_x + 1, size=[x.size(0), 1, 1], device=x.device)
    translation_y = torch.randint(-shift_y, shift_y + 1, size=[x.size(0), 1, 1], device=x.device)
    grid_batch, grid_x, grid_y = torch.meshgrid(
        torch.arange(x.size(0), dtype=torch.long, device=x.device),
        torch.arange(x.size(2), dtype=torch.long, device=x.device),
        torch.arange(x.size(3), dtype=torch.long, device=x.device),
    )
    grid_x = torch.clamp(grid_x + translation_x + 1, 0, x.size(2) + 1)
    grid_y = torch.clamp(grid_y + translation_y + 1, 0, x.size(3) + 1)
    x_pad = F.pad(x, [1, 1, 1, 1, 0, 0, 0, 0])
    x = x_pad.permute(0, 2, 3, 1).contiguous()[grid_batch, grid_x, grid_y].permute(0, 3, 1, 2).contiguous()
    return x


def rand_cutout(x, ratio=0.5):
    cutout_size = int(x.size(2) * ratio + 0.5), int(x.size(3) * ratio + 0.5)
    offset_x = torch.randint(0, x.size(2) + (1 - cutout_size[0] % 2), size=[x.size(0), 1, 1], device=x.device)
    offset_y = torch.randint(0, x.size(3) + (1 - cutout_size[1] % 2), size=[x.size(0), 1, 1], device=x.device)
    grid_batch, grid_x, grid_y = torch.meshgrid(
        torch.arange(x.size(0), dtype=torch.long, device=x.device),
        torch.arange(cutout_size[0], dtype=torch.long, device=x.device),
        torch.arange(cutout_size[1], dtype=torch.long, device=x.device),
    )
    grid_x = torch.clamp(grid_x + offset_x - cutout_size[0] // 2, min=0, max=x.size(2) - 1)
    grid_y = torch.clamp(grid_y + offset_y - cutout_size[1] // 2, min=0, max=x.size(3) - 1)
    mask = torch.ones(x.size(0), x.size(2), x.size(3), dtype=x.dtype, device=x.device)
    mask[grid_batch, grid_x, grid_y] = 0
    x = x * mask.unsqueeze(1)
    return x


AUGMENT_FNS = {
    'color': [rand_brightness, rand_saturation, rand_contrast],
    'translation': [rand_translation],
    'cutout': [rand_cutout],
}

### Experiment and training utilities

**Purpose:** Same weights/order/latent RNG in both variants. Augmentation has independent checkpointed RNG. AMP starts at loss scale1024; overflow lowers the scale and retries the same phase up to8 times, restoring BatchNorm buffers and reusing latents/augmentation. Only completed D+G rounds count. Persistent overflow, missing gradients or nonfinite FP32 fail with diagnostics. Final EMA, atomic checkpoints and exact resume are preserved.

**Inputs:** Embedded implementation below; RUNTIME is the destination directory.

**Outputs:** An overwritten, numbered worker fragment; execute the assembly cell after all fragments.


In [ ]:
%%writefile {RUNTIME}/03.py

# ---- Controlled experiment and recovery ----
VARIANTS = ('dcgan_baseline', 'dcgan_diffaugment')
PROTOCOL = 'dcgan_diffaugment_v1'

def default_config(root='/kaggle/working'):
    return {'run_mode':'smoke','seed':42,'experiment_ids':list(VARIANTS),
      'output_root':str(Path(root)/'artifacts'/'A2'/PROTOCOL),'effective_batch':64,
      'data':{'root':None,'cache_root':str(Path(root)/'anime_cache'),'val_size':2000,'test_size':5000,'image_size':64},
      'model':{'latent':128,'width':64,'lr':2e-4,'betas':[.5,.999],'ema':.999},
      'training':{'max_steps':20000,'microbatch':64,'amp':True,'amp_init_scale':1024.,'amp_max_retries':8,'save_every':1000,'preview_every':1000,'workers':2},
      'runtime':{'devices':dict(zip(VARIANTS,[0,1])),'sequential':False,'cpu':False,'heartbeat_seconds':100},
      'augmentation':{'policy':'color,translation,cutout','upstream_sha256':'d61fed24900fea3be17e3cd10d8880f895905732fd53f1f9b3013673b36fa0e2'},
      'evaluation':{'samples':5000,'batch_size':64,'kid_subsets':100,'kid_subset_size':1000,'is_splits':10,'knn_k':3,'chunk':256,'benchmark_batches':[1,64],'benchmark_repeats':30},
      'resume_from':{v:None for v in VARIANTS},'evaluate_only':False,'pilot':False}

def resolve(c):
    c=copy.deepcopy(c)
    assert c['run_mode'] in ('smoke','full')
    assert c['experiment_ids'] and set(c['experiment_ids'])<=set(VARIANTS)
    assert c['effective_batch']==c['training']['microbatch'], 'BatchNorm: use full effective batch per forward'
    assert c['augmentation']['policy']=='color,translation,cutout'
    assert math.isfinite(c['training'].get('amp_init_scale',1024.)) and c['training'].get('amp_init_scale',1024.)>0
    assert isinstance(c['training'].get('amp_max_retries',8),int) and c['training'].get('amp_max_retries',8)>=0
    if c['run_mode']=='smoke' or c['pilot']:
        c['run_mode']='smoke'; c['training'].update(max_steps=20 if c['pilot'] else 2,save_every=1,preview_every=2)
        c['evaluation'].update(samples=16,kid_subsets=2,kid_subset_size=8,is_splits=2,knn_k=3,benchmark_batches=[1,4],benchmark_repeats=2)
    if c['pilot']: c['output_root']=str(Path(c['output_root'])/'pilot')
    if c['run_mode']=='full':
        assert c['training']['max_steps']==20000 and c['effective_batch']==64
        assert c['evaluation']['samples']==5000
    return c

def fingerprint(c,m,name):
    return digest({'protocol':PROTOCOL,'variant':name,'model':c['model'],'training':c['training'],
      'augmentation':c['augmentation'],'batch':c['effective_batch'],'seed':c['seed'],'manifest':m['hash'],
      'mode':c['run_mode'],'source':hashlib.sha256(Path(__file__).read_bytes()).hexdigest()})

def save_image(x,path,nrow=8):
    if not torch.isfinite(x).all(): raise FloatingPointError('Nonfinite generated image')
    x=(x.detach().cpu().clamp(0,1)*255).round().to(torch.uint8)
    rows=math.ceil(len(x)/nrow); h,w=x.shape[-2:]
    canvas=np.zeros((rows*h,nrow*w,3),dtype=np.uint8)
    for i,image in enumerate(x): canvas[i//nrow*h:(i//nrow+1)*h,i%nrow*w:(i%nrow+1)*w]=image.permute(1,2,0).numpy()
    Image.fromarray(canvas).save(path)

def checkpoint_save(state,run):
    run=Path(run); tmp=run/'checkpoint.tmp'; torch.save(state,tmp)
    # Verify serialization before replacing either recoverable generation.
    torch.load(tmp,map_location='cpu',weights_only=False)
    last=run/'last.pt'; previous=run/'previous.pt'
    if last.exists():
        try: old=torch.load(last,map_location='cpu',weights_only=False)
        except Exception: old=None
        if old and old.get('step')!=state.get('step'): last.replace(previous)
    tmp.replace(last)

def checkpoint_load(path,expected):
    path=Path(path); errors=[]
    for candidate in (path,path.with_name('previous.pt')):
        if not candidate.exists(): continue
        try: state=torch.load(candidate,map_location='cpu',weights_only=False)
        except Exception as error: errors.append(str(error)); continue
        if state.get('fingerprint')!=expected: raise ValueError('Checkpoint incompatible with model/config/data/source')
        return state
    raise RuntimeError('No valid checkpoint: '+'; '.join(errors))

def augment(x,name,aug_state):
    if name=='dcgan_baseline': return x,aug_state
    # Reference uses global RNG. Swap only during augmentation and restore latent RNG.
    original=rng_state(); restore_rng(aug_state)
    try:
        result=DiffAugment(x,'color,translation,cutout'); advanced=rng_state()
    finally: restore_rng(original)
    return result,advanced

def build_models(c,device):
    seed_all(c['seed'])
    g=Generator(c['model']['latent'],c['model']['width']).to(device); g.apply(initialize)
    d=Discriminator(c['model']['width']).to(device); d.apply(initialize)
    return g,d

def finite_grad(model):
    gradients=[(name,p.grad) for name,p in model.named_parameters() if p.requires_grad]
    missing=[name for name,grad in gradients if grad is None]
    if not gradients or missing: raise RuntimeError('Missing gradient: '+', '.join(missing[:4]))
    bad=[name for name,grad in gradients if not torch.isfinite(grad).all()]
    if bad: raise FloatingPointError('Nonfinite gradient in '+', '.join(bad[:4]))

def optimizer_phase(model,optimizer,scaler,forward_loss,max_retries,phase):
    """Retry scaled backward overflow before committing this optimizer phase.

    No parameters/optimizer state change on overflow: GradScaler.step skips it.
    Restore BatchNorm buffers before replay. The closure reuses its latent and
    augmentation RNG; only the successful attempt advances the outer data/RNG.
    Missing gradients and FP32/nonfinite forward failures remain hard errors.
    """
    buffers={name:value.detach().clone() for name,value in model.named_buffers()}
    for attempt in range(max_retries+1):
        if attempt:
            with torch.no_grad():
                for name,value in model.named_buffers(): value.copy_(buffers[name])
        optimizer.zero_grad(set_to_none=True)
        loss,extra=forward_loss()
        if not torch.isfinite(loss): raise FloatingPointError(f'{phase}: nonfinite forward loss; try AMP=False for both variants in a fresh run')
        scaler.scale(loss).backward(); scaler.unscale_(optimizer)
        try: finite_grad(model)
        except FloatingPointError as error:
            if not scaler.is_enabled(): raise FloatingPointError(f'{phase} FP32: {error}') from error
            scale=scaler.get_scale()
            scaler.step(optimizer)  # Found inf/NaN: skip parameter and Adam updates.
            scaler.update()         # Back off loss scale before replaying this phase.
            print(f'{phase} AMP overflow: scale {scale:g} -> {scaler.get_scale():g}; retry {attempt+1}/{max_retries}; {error}',flush=True)
            if attempt==max_retries:
                raise FloatingPointError(f'{phase}: AMP overflow persisted after {max_retries} retries; try training.amp=False for BOTH variants in a fresh output_root') from error
            continue
        scaler.step(optimizer); scaler.update()
        return loss.item(),extra

def update(c,name,g,d,og,od,sg,sd,real,device,latent,aug_state):
    n=len(real); zdim=c['model']['latent']; amp=c['training']['amp'] and device.type=='cuda'
    g.train(); d.train().requires_grad_(True)
    retries=c['training'].get('amp_max_retries',8)
    z=torch.randn(n,zdim,1,1,generator=latent,device=device)
    with torch.no_grad(): fake=g(z)
    x,aug_state=augment(torch.cat([real,fake.detach()]),name,aug_state)
    def d_loss():
        with torch.amp.autocast('cuda',enabled=amp):
            a,b=d(x).chunk(2)
            loss=nn.functional.binary_cross_entropy_with_logits(a.float(),torch.ones_like(a).float())+nn.functional.binary_cross_entropy_with_logits(b.float(),torch.zeros_like(b).float())
        return loss,None
    dl,_=optimizer_phase(d,od,sd,d_loss,retries,'D')
    d.eval().requires_grad_(False)
    z=torch.randn(n,zdim,1,1,generator=latent,device=device)
    def g_loss():
        with torch.amp.autocast('cuda',enabled=amp):
            fake=g(z); augmented,next_aug=augment(fake,name,aug_state); logits=d(augmented)
            loss=nn.functional.binary_cross_entropy_with_logits(logits.float(),torch.ones_like(logits).float())
        return loss,next_aug
    gl,aug_state=optimizer_phase(g,og,sg,g_loss,retries,'G')
    return gl,dl,aug_state

@torch.no_grad()
def generated(c,g,count,seed,device):
    gen=torch.Generator(device=device).manual_seed(seed)
    g.eval(); return g(torch.randn(count,c['model']['latent'],1,1,generator=gen,device=device)).float()

def train_variant(c,m,name,run,device,stop):
    fp=fingerprint(c,m,name); marker=run/'trained.json'
    if marker.exists() and not c['evaluate_only']:
        completed=json.loads(marker.read_text())
        if completed.get('fingerprint')!=fp: raise ValueError('Use a fresh output_root for a new protocol')
        if completed.get('completed_training') and completed.get('steps')==c['training']['max_steps']:
            checkpoint=checkpoint_load(run/'last.pt',fp)
            if checkpoint['step']==c['training']['max_steps']: return
    g,d=build_models(c,device); ema=EMA(g,c['model']['ema'])
    kwargs={'lr':c['model']['lr'],'betas':tuple(c['model']['betas'])}
    og=torch.optim.Adam(g.parameters(),**kwargs); od=torch.optim.Adam(d.parameters(),**kwargs)
    enabled=c['training']['amp'] and device.type=='cuda'
    scale=c['training'].get('amp_init_scale',1024.)
    sg=torch.amp.GradScaler('cuda',enabled=enabled,init_scale=scale); sd=torch.amp.GradScaler('cuda',enabled=enabled,init_scale=scale)
    latent=torch.Generator(device=device).manual_seed(c['seed']+10)
    initial_rng=rng_state(); seed_all(c['seed']+20); aug_state=rng_state(); restore_rng(initial_rng)
    step=epoch=offset=0; history=[]; pending=None
    resume=c['resume_from'].get(name)
    if resume:
        state=checkpoint_load(run/'last.pt' if resume=='last' else resume,fp)
        g.load_state_dict(state['g']); d.load_state_dict(state['d']); ema.model.load_state_dict(state['ema'])
        og.load_state_dict(state['og']); od.load_state_dict(state['od']); sg.load_state_dict(state['sg']); sd.load_state_dict(state['sd'])
        step,epoch,offset=state['step'],state['epoch'],state['offset']; history=state['history']
        latent.set_state(state['latent']); aug_state=state['augmentation_rng']; pending=state['rng']
    elif (run/'last.pt').exists(): raise ValueError('Incomplete run exists: set resume_from[variant]="last" or use a new output_root')
    def save():
        checkpoint_save(dict(g=g.state_dict(),d=d.state_dict(),ema=ema.model.state_dict(),og=og.state_dict(),od=od.state_dict(),sg=sg.state_dict(),sd=sd.state_dict(),step=step,epoch=epoch,offset=offset,history=history,latent=latent.get_state(),augmentation_rng=aug_state,rng=rng_state(),fingerprint=fp,config=c),run)
        if history:
            with (run/'history.csv').open('w',newline='') as f:
                writer=csv.DictWriter(f,fieldnames=list(history[0])); writer.writeheader(); writer.writerows(history)
    start=time.monotonic(); elapsed=history[-1]['seconds'] if history else 0
    if device.type=='cuda': torch.cuda.reset_peak_memory_stats()
    while step<c['training']['max_steps'] and not stop.is_set():
        dataset=Images(m,'train',c['seed'],epoch)
        loader=DataLoader(dataset,batch_size=c['effective_batch'],shuffle=True,drop_last=True,num_workers=c['training']['workers'],generator=torch.Generator().manual_seed(c['seed']+epoch))
        if not len(loader): raise ValueError('Training split smaller than effective batch')
        for batch_index,images in enumerate(loader):
            if batch_index<offset: continue
            if pending is not None: restore_rng(pending); pending=None
            gl,dl,aug_state=update(c,name,g,d,og,od,sg,sd,images.to(device),device,latent,aug_state)
            ema.update(g); step+=1; offset=batch_index+1
            validation=None
            if step%c['training']['save_every']==0:
                with torch.no_grad():
                    val=next(iter(DataLoader(Images(m,'val',c['seed']),batch_size=c['effective_batch']))).to(device)
                    d.eval(); validation=d(val).sigmoid().mean().item()
            row=dict(step=step,epoch=epoch,g_loss=gl,d_loss=dl,g_loss_scale=sg.get_scale(),d_loss_scale=sd.get_scale(),validation_real_score=validation,seconds=elapsed+time.monotonic()-start,real_images_seen=step*c['effective_batch'])
            history.append(row); atomic_json(row,run/'progress.json')
            print(name,row,flush=True)
            if step%c['training']['save_every']==0: save()
            if step%c['training']['preview_every']==0:
                save_image((generated(c,ema.model,64,c['seed']+123,device)+1)/2,run/'samples.png')
            if step>=c['training']['max_steps'] or stop.is_set(): break
        else: epoch+=1; offset=0
    save()
    if step!=c['training']['max_steps']: raise InterruptedError('Paused at update boundary; resume last.pt')
    import resource
    seconds=history[-1]['seconds']
    atomic_json(dict(fingerprint=fp,completed_training=True,steps=step,real_images_seen=step*c['effective_batch'],training_seconds=seconds,projected_20000_updates_seconds=seconds*20000/step,updates_per_second=step/seconds,real_images_per_second=step*c['effective_batch']/seconds,peak_host_memory_mb=resource.getrusage(resource.RUSAGE_SELF).ru_maxrss/1024,peak_training_gpu_memory_mb=torch.cuda.max_memory_allocated()/2**20 if device.type=='cuda' else 0),marker)

### Metrics and comparison utilities

**Purpose:** FID float64, unbiased polynomial KID, IS, chunked kNN precision/recall. Images enter one Inception extractor as RGB uint8. Cache keys include manifest/checkpoint/extractor/protocol hashes. Smoke never publishes conclusions.

**Inputs:** Embedded implementation below; RUNTIME is the destination directory.

**Outputs:** An overwritten, numbered worker fragment; execute the assembly cell after all fragments.


In [ ]:
%%writefile {RUNTIME}/04.py

# ---- Metrics from one fixed Inception feature/logit contract ----
def to_uint8(x): return ((x.detach().clamp(-1,1)+1)*127.5).round().to(torch.uint8)

def kid(real,fake,subsets=100,size=1000,seed=42):
    rng=np.random.default_rng(seed); real=np.asarray(real,dtype=np.float64); fake=np.asarray(fake,dtype=np.float64)
    if size<2 or size>min(len(real),len(fake)): raise ValueError('KID subset size invalid')
    values=[]; dim=real.shape[1]
    for _ in range(subsets):
        x=real[rng.choice(len(real),size,False)]; y=fake[rng.choice(len(fake),size,False)]
        xx=(x@x.T/dim+1)**3; yy=(y@y.T/dim+1)**3; xy=(x@y.T/dim+1)**3
        values.append((xx.sum()-np.trace(xx)+yy.sum()-np.trace(yy))/(size*(size-1))-2*xy.mean())
    return float(np.mean(values)),float(np.std(values)),values

def distances(x,y):
    return (x.square().sum(1)[:,None]+y.square().sum(1)[None,:]-2*x@y.T).clamp_min(0)

def precision_recall(real,fake,k=3,chunk=256):
    real=torch.as_tensor(real,dtype=torch.float64); fake=torch.as_tensor(fake,dtype=torch.float64)
    def radii(x):
        if len(x)<=k: raise ValueError('Not enough features for kNN')
        result=[]
        for start in range(0,len(x),chunk):
            d=distances(x[start:start+chunk],x)
            d[torch.arange(len(d)),torch.arange(start,start+len(d))]=float('inf')
            result.append(d.topk(k,largest=False).values[:,-1])
        return torch.cat(result)
    rr,fr=radii(real),radii(fake)
    def coverage(query,reference,radius):
        covered=[]
        for start in range(0,len(query),chunk):
            covered.append((distances(query[start:start+chunk],reference)<=radius[None,:]).any(1))
        return torch.cat(covered).double().mean().item()
    return coverage(fake,real,rr),coverage(real,fake,fr)

def fid(real,fake):
    x=np.asarray(real,dtype=np.float64); y=np.asarray(fake,dtype=np.float64)
    a,b=np.cov(x,rowvar=False),np.cov(y,rowvar=False)
    eigen,vectors=np.linalg.eigh(a); sqrt_a=(vectors*np.sqrt(eigen.clip(0)))@vectors.T
    cross=np.linalg.eigvalsh(sqrt_a@b@sqrt_a).clip(0)
    return float(max(0,np.square(x.mean(0)-y.mean(0)).sum()+np.trace(a)+np.trace(b)-2*np.sqrt(cross).sum()))

def inception_score(logits,splits=10):
    x=torch.as_tensor(logits,dtype=torch.float64).softmax(1).clamp_min(1e-16); scores=[]
    for part in torch.tensor_split(x,splits):
        if not len(part): raise ValueError('Empty IS split')
        scores.append(((part*(part.log()-part.mean(0).log())).sum(1).mean()).exp().item())
    return float(np.mean(scores)),float(np.std(scores))

def extract(c,m,g,run,device,checkpoint_hash):
    import importlib.metadata
    from torch_fidelity.feature_extractor_inceptionv3 import FeatureExtractorInceptionV3
    extractor=FeatureExtractorInceptionV3('inception-v3-compat',['2048','logits_unbiased']).to(device).eval()
    weights=digest({n:hashlib.sha256(t.cpu().numpy().tobytes()).hexdigest() for n,t in extractor.state_dict().items()})
    protocol=dict(extractor='inception-v3-compat',weights_sha256=weights,torch_fidelity=importlib.metadata.version('torch-fidelity'),input='RGB_uint8_round_clamp_minus1_1_to_0_255',samples=c['evaluation']['samples'],evaluation=c['evaluation'],seed=c['seed']+2026,precision='FP32',manifest=m['hash'],source_sha256=hashlib.sha256(Path(__file__).read_bytes()).hexdigest())
    atomic_json(protocol,run/'metric_protocol.json'); cache=Path(c['data']['cache_root'])/'features'; cache.mkdir(exist_ok=True)
    n=c['evaluation']['samples']; batch=c['evaluation']['batch_size']
    def features(role):
        key=digest({'protocol':protocol,'role':role,'checkpoint':checkpoint_hash if role=='generated' else None})
        path=cache/(key+'.npz')
        if path.exists():
            a=np.load(path); return a['features'],a['logits']
        fs=[]; ls=[]; dataset=Images(m,role,c['seed']) if role!='generated' else None
        if dataset is not None: dataset.train=False
        if dataset is not None and len(dataset)<n: raise ValueError('Not enough real images')
        for i in range(0,n,batch):
            count=min(batch,n-i)
            x=torch.stack([dataset[j] for j in range(i,i+count)]).to(device) if dataset is not None else generated(c,g,count,c['seed']+2026+i,device)
            with torch.no_grad(): f,l=extractor(to_uint8(x))
            fs.append(f.cpu().numpy()); ls.append(l.cpu().numpy())
        f,l=np.concatenate(fs),np.concatenate(ls)
        tmp=path.with_suffix('.tmp.npz'); np.savez(tmp,features=f,logits=l); tmp.replace(path)
        return f,l
    return features('test'),features('generated'),features('train'),protocol

def evaluate(c,m,name,run,device):
    atomic_json({'state':'running'},run/'evaluation_status.json')
    fp=fingerprint(c,m,name); state=checkpoint_load(run/'last.pt',fp)
    if state['step']!=c['training']['max_steps']: raise ValueError('Training budget incomplete')
    g,_=build_models(c,device); g.load_state_dict(state['ema']); g.eval()
    with torch.no_grad():
        grid=generated(c,g,64,c['seed']+123,device); save_image((grid+1)/2,run/'samples.png')
        gen=torch.Generator(device=device).manual_seed(c['seed']+33)
        ends=torch.randn(2,c['model']['latent'],1,1,device=device,generator=gen)
        z=torch.stack([(1-a)*ends[0]+a*ends[1] for a in torch.linspace(0,1,8,device=device)])
        save_image((g(z)+1)/2,run/'interpolation.png')
    trained=json.loads((run/'trained.json').read_text()) if (run/'trained.json').exists() else {}
    checkpoint_hash=hashlib.sha256((run/'last.pt').read_bytes()).hexdigest()
    metrics={**trained,'run_mode':c['run_mode'],'completed_training':True,'fingerprint':fp,'checkpoint_sha256':checkpoint_hash,'generated_samples':c['evaluation']['samples'],'parameters':sum(p.numel() for p in g.parameters())}
    if c['run_mode']=='full':
        (real,_),(fake,logits),(training,_),protocol=extract(c,m,g,run,device,checkpoint_hash)
        e=c['evaluation']; km,ks,values=kid(real,fake,e['kid_subsets'],e['kid_subset_size'],c['seed'])
        precision,recall=precision_recall(real,fake,e['knn_k'],e['chunk']); im,istd=inception_score(logits,e['is_splits'])
        metrics.update(fid=fid(real,fake),kid_mean=km,kid_std=ks,kid_scale=1,inception_score_mean=im,inception_score_split_std=istd,generative_precision=precision,generative_recall=recall,metric_protocol_hash=digest(protocol))
        atomic_json(values,run/'kid_subsets.json')
        # Nearest neighbours are diagnostics, not proof that memorization is absent.
        nearest_train=distances(torch.tensor(fake[:16]),torch.tensor(training)).argmin(1)
        nearest_test=distances(torch.tensor(fake[:16]),torch.tensor(real)).argmin(1)
        trainset=Images(m,'train',c['seed']); trainset.train=False; testset=Images(m,'test',c['seed'])
        generated16=generated(c,g,min(c['evaluation']['batch_size'],c['evaluation']['samples']),c['seed']+2026,device)[:16].cpu()
        comparison=torch.stack([image for i in range(16) for image in (generated16[i],trainset[int(nearest_train[i])],testset[int(nearest_test[i])])])
        save_image((comparison+1)/2,run/'nearest_neighbors.png',nrow=3)
    if not all(math.isfinite(metrics[k]) for k in ('fid','kid_mean','kid_std','inception_score_mean','inception_score_split_std','generative_precision','generative_recall') if k in metrics): raise FloatingPointError('Nonfinite metrics')
    atomic_json(metrics,run/'metrics.json')
    import matplotlib; matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    plt.plot([x['step'] for x in state['history']],[x['g_loss'] for x in state['history']],label='G')
    plt.plot([x['step'] for x in state['history']],[x['d_loss'] for x in state['history']],label='D')
    plt.legend(); plt.xlabel('Updates'); plt.ylabel('Loss'); plt.tight_layout(); plt.savefig(run/'curves.png'); plt.close()
    atomic_json({'state':'complete','fingerprint':fp},run/'evaluation_status.json')

def benchmark(c,m,name,run,device):
    atomic_json({'state':'running'},run/'benchmark_status.json')
    state=checkpoint_load(run/'last.pt',fingerprint(c,m,name)); g,_=build_models(c,device); g.load_state_dict(state['ema']); g.eval()
    checkpoint_hash=hashlib.sha256((run/'last.pt').read_bytes()).hexdigest()
    result={}
    for batch in c['evaluation']['benchmark_batches']:
        z=torch.randn(batch,c['model']['latent'],1,1,device=device)
        with torch.no_grad():
            for _ in range(5): g(z)
            if device.type=='cuda': torch.cuda.synchronize(); torch.cuda.reset_peak_memory_stats()
            start=time.perf_counter()
            for _ in range(c['evaluation']['benchmark_repeats']): g(z)
            if device.type=='cuda': torch.cuda.synchronize()
            elapsed=time.perf_counter()-start; images=batch*c['evaluation']['benchmark_repeats']
        result[str(batch)]={'fingerprint':state['fingerprint'],'checkpoint_sha256':checkpoint_hash,'latency_ms_per_image':elapsed*1000/images,'images_per_second':images/elapsed,'peak_gpu_memory_mb':torch.cuda.max_memory_allocated()/2**20 if device.type=='cuda' else 0,'precision':'FP32','device':torch.cuda.get_device_name() if device.type=='cuda' else 'cpu'}
    atomic_json(result,run/'benchmark.json')
    atomic_json({'state':'complete','fingerprint':state['fingerprint']},run/'benchmark_status.json')

REQUIRED=('fid','kid_mean','kid_std','inception_score_mean','inception_score_split_std','generative_precision','generative_recall','training_seconds','updates_per_second','real_images_per_second','peak_host_memory_mb','peak_training_gpu_memory_mb')
def conclusion(rows):
    if set(rows)!=set(VARIANTS): return 'incomplete', 'Chưa đủ hai biến thể.'
    for row in rows.values():
        if row.get('run_mode')!='full' or row.get('steps')!=20000 or row.get('generated_samples')!=5000 or not row.get('metric_protocol_hash') or not row.get('completed_training') or any(k not in row or not isinstance(row[k],(int,float)) or not math.isfinite(row[k]) for k in REQUIRED) or not row.get('benchmark_complete'):
            return 'incomplete','Chưa đủ budget hoặc metrics/benchmark bắt buộc.'
    a,b=[rows[v] for v in VARIANTS]
    if a.get('metric_protocol_hash')!=b.get('metric_protocol_hash'): return 'incomplete','Metric protocols khác nhau.'
    if b['fid']<a['fid'] and b['kid_mean']<a['kid_mean']: return 'observed_improvement','Cải thiện được quan sát trên run seed42; xem thêm coverage và chi phí.'
    if b['fid']>a['fid'] and b['kid_mean']>a['kid_mean']: return 'baseline_better','Baseline tốt hơn trên FID và KID trong run seed42.'
    return 'mixed','Kết quả hỗn hợp/trade-off; không xác định mô hình thắng toàn diện.'

def report(c,m):
    root=Path(c['output_root'])/c['run_mode']; rows={}
    for name in VARIANTS:
        run=root/name; path=run/'metrics.json'
        if not path.exists(): continue
        row=json.loads(path.read_text())
        if row.get('fingerprint')!=fingerprint(c,m,name): continue
        status=run/'evaluation_status.json'
        if not status.exists() or json.loads(status.read_text()).get('state')!='complete': continue
        bench=run/'benchmark.json'; data=json.loads(bench.read_text()) if bench.exists() else {}
        bs=run/'benchmark_status.json'
        row['benchmark_complete']=bs.exists() and json.loads(bs.read_text()).get('state')=='complete' and all(str(n) in data and data[str(n)].get('fingerprint')==row['fingerprint'] and data[str(n)].get('checkpoint_sha256')==row.get('checkpoint_sha256') and data[str(n)].get('precision')=='FP32' and all(isinstance(data[str(n)].get(k),(int,float)) and math.isfinite(data[str(n)][k]) and data[str(n)][k]>0 for k in ('images_per_second','latency_ms_per_image')) for n in [1,64]); row['benchmark']=data; rows[name]=row
    if len(rows)==2 and len({row.get('benchmark',{}).get('1',{}).get('device') for row in rows.values()})!=1:
        for row in rows.values(): row['benchmark_complete']=False
    state,text=conclusion(rows)
    atomic_json({'protocol':PROTOCOL,'state':state,'conclusion':text,'experiments':rows},root/'comparison.json')
    lines=['# DCGAN và DiffAugment', '', text, '',
      '## Dữ liệu và protocol',
      f"Nguồn khai báo: {m['source']}, RGB64×64; manifest `{m['hash']}`. Seed{c['seed']}, Adam lr={c['model']['lr']}/betas={c['model']['betas']}, batch{c['effective_batch']}, EMA{c['model']['ema']}, budget{c['training']['max_steps']} updates. Mode: `{c['run_mode']}`.",
      'Cùng khởi tạo, thứ tự ảnh, flip và chuỗi latent; chỉ nhánh DiffAugment dùng color,translation,cutout với RNG riêng. Validation chỉ chẩn đoán, không chọn checkpoint.', '',
      '## Phương pháp từ paper',
      '[Zhao et al., NeurIPS 2020](https://arxiv.org/abs/2006.10738): augmentation khả vi trên real/fake ở D và fake ở G; gradient G xuyên transform. D weights và BatchNorm statistics đóng băng trong pha G.', '',
      '## Thực nghiệm gốc và đối chiếu',
      'Paper nghiên cứu nhiều kiến trúc và mức dữ liệu. Nhóm chỉ ablate một DCGAN trên toàn bộ train anime hợp lệ; không tuyên bố tái hiện toàn bộ paper hay chứng minh data efficiency.', '',
      '## Kết quả của nhóm']
    if state!='incomplete':
        lines+=['| Biến thể | '+' | '.join(REQUIRED)+' |','|---|'+'---|'*len(REQUIRED)]
        for name,row in rows.items(): lines+=['| '+name+' | '+' | '.join(f'{row[k]:.6g}' for k in REQUIRED)+' |']
        lines+=['','KID ở scale gốc; IS std là biến thiên giữa split, không phải giữa seed.', '',
                '| Biến thể | Batch | ms/ảnh | Ảnh/s | Peak VRAM MB | GPU |', '|---|---:|---:|---:|---:|---|']
        for name,row in rows.items():
            for batch in [1,64]:
                b=row['benchmark'][str(batch)]
                lines += [f"| {name} | {batch} | {b['latency_ms_per_image']:.4g} | {b['images_per_second']:.4g} | {b['peak_gpu_memory_mb']:.4g} | {b['device']} / FP32 |"]
    else:
        lines+=['Chưa công bố bảng so sánh chính: cần hai full run đủ20.000 updates, metrics5.000 mẫu, cùng protocol và benchmark. Smoke/pilot chỉ kiểm tra pipeline.']
        (root/'comparison.csv').unlink(missing_ok=True)
    lines+=['','## Hạn chế và thảo luận',
      'Một seed không chứng minh statistical significance. Inception được học từ ImageNet nên có thiên lệch khi đánh giá anime. Precision/Recall đo trong feature space; nearest neighbours không chứng nhận không ghi nhớ. Chi phí train đồng thời chịu ảnh hưởng CPU/I/O.',
      'Bổ sung phân tích failure cases, độ phủ và chi phí từ kết quả thật; không chọn winner nếu FID/KID mâu thuẫn. PDF/video cuối chưa được tạo tự động.', '', '## Hình và provenance']
    for name,row in rows.items():
        lines += [f"### {name}",f"Checkpoint SHA256: `{row.get('checkpoint_sha256','pending')}`; metric protocol: `{row.get('metric_protocol_hash','pending')}`."]
        for figure in ['samples','interpolation','curves','nearest_neighbors']:
            if (root/name/(figure+'.png')).exists():lines+=[f'![{name} {figure}]({name}/{figure}.png)']
    (root/'report.md').write_text('\n'.join(lines)+'\n')
    if state!='incomplete':
        with (root/'comparison.csv').open('w',newline='') as f:
            keys=['variant',*REQUIRED]; writer=csv.DictWriter(f,fieldnames=keys); writer.writeheader()
            for name,row in rows.items(): writer.writerow({'variant':name,**{k:row[k] for k in REQUIRED}})
    return state,text

### Worker command entrypoint

**Purpose:** Independent train/evaluate/benchmark actions. Heartbeat every 100 seconds, safe-stop request and explicit failure JSON; metric failure preserves trained checkpoints.

**Inputs:** Embedded implementation below; RUNTIME is the destination directory.

**Outputs:** An overwritten, numbered worker fragment; execute the assembly cell after all fragments.


In [ ]:
%%writefile {RUNTIME}/05.py

def main():
    import argparse,traceback,platform,resource
    p=argparse.ArgumentParser(); p.add_argument('--config',required=True); p.add_argument('--variant',choices=VARIANTS,required=True); p.add_argument('--action',choices=['train','evaluate','benchmark'],default='train'); args=p.parse_args()
    c=json.loads(Path(args.config).read_text()); name=args.variant; run=Path(c['output_root'])/c['run_mode']/name; run.mkdir(parents=True,exist_ok=True)
    stop=threading.Event(); done=threading.Event()
    signal.signal(signal.SIGTERM,lambda *_:stop.set()); signal.signal(signal.SIGINT,lambda *_:stop.set())
    def heartbeats():
        while not done.wait(c['runtime']['heartbeat_seconds']):
            with (run/'heartbeat.jsonl').open('a') as f:f.write(json.dumps({'time':time.time(),'pid':os.getpid(),'variant':name,'action':args.action})+'\n')
            print('heartbeat',name,time.strftime('%H:%M:%S'),flush=True)
    threading.Thread(target=heartbeats,daemon=True).start()
    action='evaluate' if c['evaluate_only'] and args.action=='train' else args.action
    try:
        device=torch.device('cpu' if c['runtime']['cpu'] or not torch.cuda.is_available() else 'cuda:0')
        if c['run_mode']=='full' and device.type!='cuda': raise RuntimeError('Full runs require CUDA')
        if device.type=='cpu': torch.set_num_threads(2)
        m=json.loads((Path(c['data']['cache_root'])/'manifest.json').read_text())
        import importlib.metadata
        versions={}
        for package in ['numpy','Pillow','scipy','matplotlib','torch-fidelity']:
            try: versions[package]=importlib.metadata.version(package)
            except importlib.metadata.PackageNotFoundError: versions[package]=None
        atomic_json({**c,'assignment':'A2','experiment_id':name,'protocol':PROTOCOL},run/'config.json'); atomic_json({'torch':torch.__version__,'packages':versions,'python':platform.python_version(),'cuda':torch.version.cuda,'visible_devices':os.getenv('CUDA_VISIBLE_DEVICES'),'device':torch.cuda.get_device_name(0) if device.type=='cuda' else 'cpu'},run/'environment.json')
        if action=='train': train_variant(c,m,name,run,device,stop)
        elif action=='evaluate': evaluate(c,m,name,run,device)
        else: benchmark(c,m,name,run,device)
    except BaseException as error:
        atomic_json({'state':'failed','action':action,'error':str(error),'traceback':traceback.format_exc()},run/('evaluation_status.json' if action=='evaluate' else action+'_status.json'))
        raise
    finally: done.set()

if __name__=='__main__': main()

### Assemble worker

**Purpose:** Overwrite fragments and concatenate in order. No append duplication and no repository runtime import.

**Inputs:** The six visible runtime fragments and RUNTIME path.

**Outputs:** Standalone worker.py and the study API; no GPU model allocated.


In [ ]:
WORKER=RUNTIME/'worker.py'
WORKER.write_text('\n'.join((RUNTIME/f).read_text() for f in ['00.py', '01.py', '02.py', '03.py', '04.py', '05.py']))
import importlib.util
spec=importlib.util.spec_from_file_location('study',WORKER)
study=importlib.util.module_from_spec(spec);spec.loader.exec_module(study)

## 2. Central configuration
Change these values before subsequent cells. Full budget is locked; pilot uses its own smoke namespace.


### CONFIG

**Purpose:** Default smoke; use runtime.cpu=True and sequential=True for CPU testing. To resume attach saved outputs and set resume_from per variant. evaluate_only recomputes metrics without retraining.

**Inputs:** study.default_config; edit paths, variants, mode, budgets and device assignments here.

**Outputs:** Resolved config.json; full-run metric dependencies checked.


In [ ]:
CONFIG=study.default_config(str(RUNTIME.parent))
# CONFIG['run_mode']='full'
# CONFIG['runtime'].update(cpu=True,sequential=True)
# CONFIG['pilot']=True
# CONFIG['training'].update(amp=False)  # FP32 diagnostic: apply to BOTH variants; use a fresh output_root.
# CONFIG['training'].update(amp_init_scale=1024.,amp_max_retries=8)
# CONFIG['resume_from']['dcgan_baseline']='last'
CONFIG=study.resolve(CONFIG)
if CONFIG['run_mode']=='full':
    import importlib.metadata
    if importlib.util.find_spec('torch_fidelity') is None:
        subprocess.check_call([sys.executable,'-m','pip','install','torch-fidelity==0.3.0'])
    if importlib.metadata.version('torch-fidelity')!='0.3.0':
        raise RuntimeError('Install torch-fidelity==0.3.0 and restart the kernel for a pinned metric protocol')
(RUNTIME/'config.json').write_text(json.dumps(CONFIG,indent=2))
print(json.dumps(CONFIG,indent=2))

## 3. Data, split and EDA
Keep all valid training images after validation2,000/test5,000. Source attribution: https://github.com/bchao1/Anime-Face-Dataset. Exports exclude original images/cache; upstream reports redistribution concerns.


### Prepare shared cache and EDA

**Purpose:** Validate/deduplicate once in the parent process. Record dimensions, rejected files and split hash; images are read-only in workers.

**Inputs:** CONFIG.data and seed; mounted Anime Face Dataset.

**Outputs:** Manifest/hash, read-only uint8 mmap, rejected/duplicate records and eda.png.


In [ ]:
manifest=study.prepare(CONFIG)
dataset=study.Images(manifest,'train',CONFIG['seed'])
study.save_image((study.torch.stack([dataset[i] for i in range(64)])+1)/2,RUNTIME/'eda.png')
from IPython.display import display, Image
display(Image(filename=str(RUNTIME/'eda.png')))
print({role:sum(r['split']==role for r in manifest['records']) for role in ['train','val','test']})

## 4. Smoke, pilot and full training
Smoke validates both branches and checkpoint reload. Pilot uses 20 updates, preserves the full budget,
and must be run before budgeting Kaggle sessions. Full training remains 20,000 updates per variant.
Training wall time includes checkpoint/preview overhead and concurrent CPU contention; do not interpret it as isolated GPU speed.


### Launch isolated workers

**Purpose:** One subprocess per GPU; sequential CPU/single-GPU fallback is explicit. Failed workers do not terminate siblings. Ctrl-C requests boundary checkpoints. Inspect action_status.json and heartbeat.jsonl.

**Inputs:** Resolved CONFIG, WORKER and the prepared cache/manifest.

**Outputs:** Separate logs, status, progress, heartbeat, checkpoints and trained.json; pilot cost projection.


In [ ]:
def launch(action):
    import torch
    names=CONFIG['experiment_ids']; sequential=CONFIG['runtime']['sequential'] or action!='train'
    if CONFIG['runtime']['cpu'] and not sequential: raise ValueError('CPU smoke requires sequential=True')
    if not CONFIG['runtime']['cpu']:
        ids=[CONFIG['runtime']['devices'][v] for v in names]
        if sequential: ids=[ids[0]]
        if not sequential and len(set(ids))!=len(ids): raise ValueError('Assign distinct GPUs for concurrent jobs')
        if any(i>=torch.cuda.device_count() for i in ids): raise RuntimeError('Select T4x2 or explicitly configure CPU/sequential GPU0')
    (RUNTIME/'config.json').write_text(json.dumps(CONFIG,indent=2)); jobs=[]
    try:
        for name in names:
            run=Path(CONFIG['output_root'])/CONFIG['run_mode']/name;run.mkdir(parents=True,exist_ok=True)
            env=os.environ.copy(); device=CONFIG['runtime']['devices'][names[0] if sequential else name]
            env['CUDA_VISIBLE_DEVICES']='' if CONFIG['runtime']['cpu'] else str(device)
            log=(run/(action+'.log')).open('a',buffering=1)
            try:
                p=subprocess.Popen([sys.executable,'-u',str(WORKER),'--config',str(RUNTIME/'config.json'),'--variant',name,'--action',action],env=env,stdout=log,stderr=subprocess.STDOUT)
            except OSError as error:
                log.close(); print('Launch failed:',name,error); continue
            jobs.append((name,p,log))
            if sequential: p.wait()
        while any(p.poll() is None for _,p,_ in jobs):
            print([(name,p.pid,p.poll()) for name,p,_ in jobs],flush=True);time.sleep(10)
    except KeyboardInterrupt:
        for _,p,_ in jobs:
            if p.poll() is None:p.terminate()
        for _,p,_ in jobs:p.wait()
        raise
    finally:
        for _,_,log in jobs:log.close()
    failed=[n for n,p,_ in jobs if p.returncode]+[n for n in names if n not in [j[0] for j in jobs]]
    if failed:
        for name in failed:
            run=Path(CONFIG['output_root'])/CONFIG['run_mode']/name
            path=run/(action+'.log')
            print(f'--- {name}: {action}, log={path} ---',flush=True)
            if path.exists():print(path.read_text(errors='replace')[-6000:],flush=True)
        raise RuntimeError('Worker failed: '+str(failed)+'. See worker log tails printed above; successful siblings are retained.')
launch('evaluate' if CONFIG['evaluate_only'] else 'train')
if CONFIG['pilot']:
    for name in CONFIG['experiment_ids']:
        pilot=json.loads((Path(CONFIG['output_root'])/CONFIG['run_mode']/name/'trained.json').read_text())
        print(name,{k:pilot[k] for k in ['training_seconds','projected_20000_updates_seconds','peak_training_gpu_memory_mb','peak_host_memory_mb']})
    print('Projection includes short-run overhead; validate GPU/storage limits before full training.')

## 5. Evaluation and inference benchmarks
Use 5,000 generated images and the same 5,000 real test images. Shared evaluation latents, no augmentation.
FID-2048 is primary; KID uses 100 subsets ×1,000 (raw scale1); IS uses10 splits.
Precision/Recall use k3 Inception2048 manifolds, not classifier precision/recall.
ImageNet feature bias limits anime interpretation. IS split std is not cross-seed uncertainty.
Benchmarks use FP32 batch1/64, same T4, warm-up and CUDA synchronization, separately from metrics.


### Evaluate then benchmark

**Purpose:** Only evaluate after workers finish. Failure leaves training intact. evaluate_only=True can retry metrics without retraining.

**Inputs:** Completed EMA checkpoints, CONFIG.evaluation and the shared manifest.

**Outputs:** Metrics/benchmark JSON, figures and comparison.json/report.md; full CSV only when complete.


In [ ]:
launch('evaluate')
launch('benchmark')
print(study.report(CONFIG,manifest))

## 6. Visual comparisons
Inspect samples.png (64 shared latents), interpolation.png, G/D curves and nearest_neighbors.png.
Neighbour grid columns: generated / nearest in fixed5,000 train subset / nearest in test.
This diagnoses memorization but cannot certify its absence. Do not use PSNR/SSIM as unconditional-generation quality metrics.


### Inspect and export

**Purpose:** Package trusted checkpoints, metrics and report; exclude source images and cache. Retain outputs to resume in future Kaggle sessions.

**Inputs:** Per-variant artifacts below CONFIG.output_root.

**Outputs:** Visible generated figures and an A2 artifact ZIP excluding source images/cache.


In [ ]:
root=Path(CONFIG['output_root'])/CONFIG['run_mode']
for name in CONFIG['experiment_ids']:
    for image in ['samples.png','interpolation.png','nearest_neighbors.png','curves.png']:
        p=root/name/image
        if p.exists():display(Image(filename=str(p)))
archive=Path(CONFIG['output_root'])/('A2_'+CONFIG['run_mode']+'.zip')
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
    for p in root.rglob('*'):
        if p.is_file():z.write(p,str(p.relative_to(Path(CONFIG['output_root']))))
    z.writestr('manifest.json',json.dumps({k:v for k,v in manifest.items() if k!='array'}))
print('Save Kaggle outputs or download:',archive)

## 7. Paper analysis and honest conclusions
**Method:** DiffAugment regularizes the discriminator using differentiable color, translation and cutout on real and fake. The G gradient traverses these transforms. Explain the distinction from augmenting real images only.

**Original experiments:** Zhao et al. study multiple GAN architectures/datasets, including low-data FFHQ/LSUN settings, ImageNet and reduced CIFAR training sets. Read the cited paper tables; do not copy their numbers into our experiment table.

**Limitations:** augmentation suitability, GAN instability and compute overhead; natural-image Inception bias on anime; exact dedup does not remove near duplicates or identity leakage. One seed does not establish significance.

| Paper | Our experiment |
|---|---|
| Multiple architectures/data regimes | One DCGAN architecture, entire usable anime train split |
| Differentiable real/fake augmentation | Author color,translation,cutout policy, isolated RNG |
| Broad data-efficiency claims | Controlled single-seed ablation, no claim across data scales |
| Paper-specific training recipes | Adam2e-4,20k updates,batch64,final EMA0.999 |

**Results:** read comparison.json/report.md. Only both complete full runs with all required metrics and benchmarks produce a conclusion. FID+KID agreement supports observed improvement at seed42; conflict is a trade-off. Include coverage and cost, not an unconditional winner.
PDF/video delivery remains separate from this notebook and generated Markdown.
